# RAD21 occupancy vs. optical-tweezers measurement

Correlates a per-motif biophysical measurement against cohesin (RAD21) ChIP-seq
signal at 21 candidate CTCF motifs.

**Coordinates are pre-loaded and verified.** They are **hg38**, checked base-for-base
against the reference. (The source spreadsheet's coordinates were 0-based; these have
been converted, so they are correct as written here.)

## How to use this

1. Run **Setup**.
2. In **Step 2**, paste the URL or local path of one or more RAD21 bigWigs. Nothing
   needs downloading — `pyBigWig` streams the few KB it needs from a remote URL.
3. Run everything else top to bottom (`Kernel → Restart & Run All`).

Add a CTCF bigWig too if you can — the notebook will then test whether RAD21 tells you
anything *beyond* CTCF occupancy, which is the main confound here.

## Setup

In [1]:
#%pip install pyBigWig pandas numpy scipy matplotlib
import warnings
import numpy as np
import pandas as pd
import pyBigWig
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 50)

# plot styling: recessive axes, neutral ink
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9,
    "axes.edgecolor": "#c9c8c3", "axes.linewidth": 0.8,
    "axes.labelcolor": "#0b0b0b", "text.color": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
CLASS_COLOR = {"hard": "#2a78d6", "intermediate": "#eb6834", "soft": "#1baf7a"}
print("ready")

ready


## Step 1 — Your data (pre-filled, edit if needed)

In [2]:
# hg38, BED-style 0-based half-open. Verified against the reference genome.
MOTIF_TSV = """name\tchrom\tstart\tend\tstrand\tgene\tclass
CTCF_motif_1\tchr10\t101050396\t101050415\t-\tKAZALD1\thard
CTCF_motif_2\tchr14\t24036912\t24036931\t-\tDHRS2/DHRS4\thard
CTCF_motif_3\tchr7\t151168385\t151168404\t+\tAGAP3\thard
CTCF_motif_4\tchr4\t54433873\t54433892\t-\tCHIC2/KIT/PDGFRA\thard
CTCF_motif_5\tchr2\t231412528\t231412547\t-\tARMC9/B3GNT7\thard
CTCF_motif_6\tchr1\t156892436\t156892455\t+\tARHGEF11/ETV3\thard
CTCF_motif_7\tchr19\t18469755\t18469774\t+\tCERS1/CRLF1\thard
CTCF_motif_8\tchr17\t57783440\t57783459\t-\tMSI2\thard
CTCF_motif_9\tchr2\t85266645\t85266664\t+\tTCF7L1\tintermediate
CTCF_motif_10\tchr3\t127544419\t127544438\t-\tPLXNA1/ZXDC\thard
CTCF_motif_11\tchr19\t18386258\t18386277\t+\tCCDC124/ELL\thard
CTCF_motif_12\tchr4\t54441034\t54441053\t-\tKIT\thard
CTCF_motif_13\tchr12\t52879676\t52879695\t-\tNR4A1\thard
CTCF_motif_14\tchr1\t201448268\t201448287\t+\tCSRP1/PHLDA3\tintermediate
CTCF_motif_15\tchr18\t48676951\t48676970\t+\tCTIF\tsoft
CTCF_motif_16\tchr11\t65780047\t65780066\t+\tBANF1/CFL1\thard
CTCF_motif_17\tchr11\t69919001\t69919020\t+\tCCND1/FGF3/FGF4\thard
CTCF_motif_18\tchr12\t6038323\t6038342\t-\tNTF3/VWF\thard
CTCF_motif_19\tchr1\t201482238\t201482257\t+\tCSRP1\tintermediate
CTCF_motif_20\tchr15\t63487478\t63487497\t-\tUSP3\thard
CTCF_motif_21\tchr11\t75205114\t75205133\t-\tSLCO2B1\thard"""

import io
motifs = pd.read_csv(io.StringIO(MOTIF_TSV), sep="\t")
motifs["mid"] = (motifs.start + motifs.end) // 2
motifs["num"] = motifs.name.str.replace("CTCF_motif_", "", regex=False).astype(int)
print(f"{len(motifs)} motifs   classes: {motifs['class'].value_counts().to_dict()}")
motifs.head(3)

21 motifs   classes: {'hard': 17, 'intermediate': 3, 'soft': 1}


,name,chrom,start,end,strand,gene,class,mid,num
0,CTCF_motif_1,chr10,101050396,101050415,-,KAZALD1,hard,101050405,1
1,CTCF_motif_2,chr14,24036912,24036931,-,DHRS2/DHRS4,hard,24036921,2
2,CTCF_motif_3,chr7,151168385,151168404,+,AGAP3,hard,151168394,3


### Your tweezers measurements

Pre-filled from the `State 2 data` sheet. **Two things to check:**

- The sheet's two blocks disagreed on motif→value assignment. The values below use the
  alignment corroborated by the second table (the one where motif 5 = 0.707, motif 18 =
  2.456), which is the one that cross-checks.
- The two tables give **different State 2 values for the same motif** — motif 5 is 0.773
  in one and 0.52 in the other. Both are below as `state2_a` and `state2_b`. Set
  `X_COLUMN` to whichever is current.

`PDGFRA` and `Consensus` rows from the sheet are not among the 21 motifs, so they're
excluded here.

In [4]:
# motif number -> (state2 first table, state2 second table)
TWEEZERS = {
    2:  (0.796, 0.636),
    4:  (0.774,  0.710),
    5:  (0.773, 0.706),
    10: (0.445,  0.636),
    11: (0.748, None),
    12: (0.642,  0.694),
    15: (0.729, None),
    18: (0.675, 0.689),
    21: (0.460,  0.469),
}

# RAD21 numbers already in your sheet, kept for comparison with the new extraction
PRIOR_RAD21 = {
    2: (0.323938, None), 4: (None, 4.0), 5: (0.706960, 4.7), 10: (None, 5.0),
    11: (1.235948, None), 12: (1.359777, 5.6), 15: (2.080045, None),
    18: (2.455691, 11.6), 21: (5.370311, 30.9),
}

X_COLUMN = "state2_a"   # <<< "state2_a" or "state2_b"

motifs["state2_a"] = motifs.num.map(lambda n: TWEEZERS.get(n, (None, None))[0])
motifs["state2_b"] = motifs.num.map(lambda n: TWEEZERS.get(n, (None, None))[1])
motifs["prior_rad21_calc"] = motifs.num.map(lambda n: PRIOR_RAD21.get(n, (None, None))[0])
motifs["prior_rad21_hekmax"] = motifs.num.map(lambda n: PRIOR_RAD21.get(n, (None, None))[1])

n_a, n_b = motifs.state2_a.notna().sum(), motifs.state2_b.notna().sum()
print(f"state2_a: {n_a} motifs    state2_b: {n_b} motifs    using: {X_COLUMN}")

def r_critical(n, alpha=0.05):
    """Smallest |r| that reaches significance at this n (two-tailed)."""
    if n < 4:
        return float("nan")
    t = stats.t.ppf(1 - alpha / 2, n - 2)
    return t / np.sqrt(t**2 + n - 2)

n_now = int(motifs[X_COLUMN].notna().sum())
print(f"\nPower: at n={n_now} you need |rho| >= {r_critical(n_now):.2f} for p<0.05")
for n in (n_now, 12, 21):
    print(f"   n={n:2d}  ->  |rho| >= {r_critical(n):.2f}")
print("Measuring the remaining motifs buys more than any choice of ChIP metric.")

motifs[["name", "gene", "class", "state2_a", "state2_b"]].dropna(
    subset=["state2_a", "state2_b"], how="all")

state2_a: 9 motifs    state2_b: 7 motifs    using: state2_a

Power: at n=9 you need |rho| >= 0.67 for p<0.05
   n= 9  ->  |rho| >= 0.67
   n=12  ->  |rho| >= 0.58
   n=21  ->  |rho| >= 0.43
Measuring the remaining motifs buys more than any choice of ChIP metric.


,name,gene,class,state2_a,state2_b
1,CTCF_motif_2,DHRS2/DHRS4,hard,0.796,0.636
3,CTCF_motif_4,CHIC2/KIT/PDGFRA,hard,0.774,0.710
4,CTCF_motif_5,ARMC9/B3GNT7,hard,0.773,0.706
9,CTCF_motif_10,PLXNA1/ZXDC,hard,0.445,0.636
10,CTCF_motif_11,CCDC124/ELL,hard,0.748,NaN
11,CTCF_motif_12,KIT,hard,0.642,0.694
14,CTCF_motif_15,CTIF,soft,0.729,NaN
17,CTCF_motif_18,NTF3/VWF,hard,0.675,0.689
20,CTCF_motif_21,SLCO2B1,hard,0.460,0.469


## Step 2 — Point at your bigWigs  ← the only cell you must edit

Use **fold-change-over-control** bigWigs (not raw signal), on **hg38/GRCh38**.
Local paths and `https://` URLs both work; remote files are streamed, not downloaded.

**Finding files:**

- **ENCODE** — [RAD21 in HEK293, GRCh38](https://www.encodeproject.org/search/?type=Experiment&target.label=RAD21&biosample_ontology.term_name=HEK293&assembly=GRCh38).
  Open an experiment, pick the *fold change over control* bigWig, and use its download
  URL: `https://www.encodeproject.org/files/ENCFFxxxxxxx/@@download/ENCFFxxxxxxx.bigWig`
- **ChIP-Atlas** — [chip-atlas.dbcls.jp](https://chip-atlas.dbcls.jp/) → antigen `RAD21`,
  cell type `HEK293`. Per-experiment bigWigs follow
  `https://chip-atlas.dbcls.jp/data/hg38/eachData/bw/SRXxxxxxxx.bw`.
  This is the better source if you specifically need **293T** rather than 293.

Give **two or more RAD21 datasets** if you can. One experiment's fold-enrichment is a
property of that antibody and sequencing depth; you asked how much cohesin *tends* to be
at these sites, and that needs a consensus across experiments. Put `RAD21` or `CTCF` in
each label — the notebook keys off that.

In [6]:
BIGWIGS = {
    "RAD21_rep1": "https://www.encodeproject.org/files/ENCFFxxxxxxx/@@download/ENCFFxxxxxxx.bigWig",
    # "RAD21_rep2": "/path/to/local/RAD21_other_dataset.bw",
    # "CTCF_rep1":  "https://www.encodeproject.org/files/ENCFFyyyyyyy/@@download/ENCFFyyyyyyy.bigWig",
}

FLANK = 250      # bp each side of the motif midpoint
SMOOTH = 50      # bp smoothing before taking the max

if not BIGWIGS:
    print("!! No bigWigs configured — add at least one RAD21 file above.")
else:
    for lab, p in BIGWIGS.items():
        try:
            bw = pyBigWig.open(p)
            ok = bw is not None and len(bw.chroms()) > 0
            print(f"{'OK  ' if ok else 'FAIL'} {lab:16s} {len(bw.chroms()) if ok else 0} contigs")
            if ok:
                bw.close()
        except Exception as e:
            print(f"FAIL {lab:16s} {type(e).__name__}: {e}")

FAIL RAD21_rep1       RuntimeError: Received an error during file opening! Unknown error during file opening.


[bwHdrRead] There was an error while reading in the header!
[pyBwOpen] bw is NULL!


## Step 3 — Extract signal  *(run the rest as-is)*

In [ ]:
def extract(path, motifs, flank=FLANK, smooth=SMOOTH):
    """Signal summary in a window around each motif midpoint."""
    bw = pyBigWig.open(path)
    chroms = bw.chroms()
    rows = []
    for _, r in motifs.iterrows():
        c = r.chrom if r.chrom in chroms else r.chrom.replace("chr", "")
        if c not in chroms:
            rows.append((np.nan,) * 5)
            continue
        lo, hi = max(0, r.mid - flank), min(chroms[c], r.mid + flank)
        v = np.array(bw.values(c, lo, hi), dtype=float)
        v = np.nan_to_num(v, nan=0.0)
        if v.size == 0:
            rows.append((np.nan,) * 5)
            continue
        sm = np.convolve(v, np.ones(smooth) / smooth, mode="same")
        rows.append((float(sm.max()), float(v.max()), float(v.mean()),
                     float(np.median(v)), float(np.argmax(sm) + lo - r.mid)))
    bw.close()
    return pd.DataFrame(rows, columns=["max_smooth", "max_raw", "mean",
                                       "median", "summit_offset"])

res = motifs.copy()
for lab, path in BIGWIGS.items():
    sub = extract(path, motifs)
    for c in sub.columns:
        res[f"{lab}.{c}"] = sub[c].values
    print(f"extracted  {lab}")

rad_cols = [c for c in res.columns if c.endswith(".max_smooth") and "RAD21" in c.upper()]
ctcf_cols = [c for c in res.columns if c.endswith(".max_smooth") and "CTCF" in c.upper()]
print(f"\nRAD21 datasets: {len(rad_cols)}   CTCF datasets: {len(ctcf_cols)}")
res[["name", "gene", "class"] + rad_cols + ctcf_cols]

### QC — look at this before trusting any correlation

`summit_offset` is where the local signal maximum sits relative to your motif. If it's
near the window edge (±250), there is no peak centred on that motif and the "signal" is
spillover from something nearby. Those sites should probably be dropped, not scored.

In [ ]:
off_cols = [c for c in res.columns if c.endswith(".summit_offset")]
if off_cols:
    qc = res[["name", "gene", "class"] + rad_cols + off_cols].copy()
    edge = (res[off_cols].abs() > FLANK * 0.8).any(axis=1)
    if edge.any():
        print(f"WARNING — {edge.sum()} motif(s) have a summit at the window edge.")
        print("No peak is centred on these motifs; consider excluding them.")
        display(qc[edge])
    else:
        print("OK — every motif has its signal summit well inside the window.")
    print("\nAll sites:")
    display(qc)

# do independent RAD21 datasets even agree with each other?
if len(rad_cols) > 1:
    print("\nBetween-dataset agreement (Spearman) — this caps any correlation you can find:")
    display(res[rad_cols].corr(method="spearman").round(2))

In [ ]:
# Consensus RAD21: mean of within-dataset rank percentiles.
# Ranks, not raw fold-enrichment, because absolute FE is not comparable across experiments.
if rad_cols:
    res["RAD21_consensus_pct"] = res[rad_cols].rank(pct=True).mean(axis=1)
    if len(rad_cols) == 1:
        res["RAD21_consensus_pct"] = res[rad_cols[0]].rank(pct=True)
if ctcf_cols:
    res["CTCF_consensus_pct"] = res[ctcf_cols].rank(pct=True).mean(axis=1)

show = ["name", "gene", "class", X_COLUMN] + rad_cols
if rad_cols:
    show += ["RAD21_consensus_pct"]
show += ["prior_rad21_calc", "prior_rad21_hekmax"]
res[show].sort_values("RAD21_consensus_pct" if rad_cols else "name", ascending=False)

## Step 4 — Correlations

In [ ]:
def corr_table(df, xcol, metrics):
    out = []
    for m in metrics:
        v = df[[m, xcol]].dropna()
        if len(v) < 4:
            continue
        rho, p_s = stats.spearmanr(v[m], v[xcol])
        r, p_p = stats.pearsonr(np.log2(v[m] + 0.1), v[xcol])
        out.append(dict(metric=m, n=len(v), spearman_rho=round(rho, 3),
                        spearman_p=round(p_s, 4), pearson_log_r=round(r, 3),
                        pearson_p=round(p_p, 4)))
    return pd.DataFrame(out)

metrics = [c for c in res.columns
           if c.endswith((".max_smooth", ".max_raw", ".mean"))
           or c in ("RAD21_consensus_pct", "CTCF_consensus_pct")]
metrics += [c for c in ("prior_rad21_calc", "prior_rad21_hekmax") if res[c].notna().any()]

ct = corr_table(res, X_COLUMN, metrics)
print(f"X = {X_COLUMN}\n")
print("Spearman is the one to read — ChIP fold-enrichment is heavy-tailed.")
print("Pick ONE metric as primary before interpreting; scanning all of them at this")
print("n will produce a 'significant' hit by chance.\n")
ct

In [ ]:
# Does RAD21 add anything beyond CTCF occupancy?
# RAD21 at CTCF sites is largely a readout of CTCF binding, so a raw RAD21
# correlation may be entirely mediated by CTCF.
if rad_cols and ctcf_cols:
    v = res[[rad_cols[0], ctcf_cols[0], X_COLUMN]].dropna()
    if len(v) >= 5:
        lr, lc = np.log2(v[rad_cols[0]] + 0.1), np.log2(v[ctcf_cols[0]] + 0.1)
        resid = lr - np.polyval(np.polyfit(lc, lr, 1), lc)
        rho_raw, p_raw = stats.spearmanr(lr, v[X_COLUMN])
        rho_res, p_res = stats.spearmanr(resid, v[X_COLUMN])
        rho_cc, _ = stats.spearmanr(lr, lc)
        print(f"n = {len(v)}")
        print(f"RAD21 vs CTCF (collinearity)      rho = {rho_cc:+.3f}")
        print(f"X vs RAD21 (raw)                  rho = {rho_raw:+.3f}  p = {p_raw:.4f}")
        print(f"X vs RAD21 | CTCF (residualized)  rho = {rho_res:+.3f}  p = {p_res:.4f}")
        print("\nIf the residualized value collapses toward 0, what you are seeing is")
        print("CTCF occupancy, not a cohesin-specific effect.")
    else:
        print(f"Only {len(v)} complete cases — need >=5.")
else:
    print("Add a CTCF bigWig to run this. It is the most important control here.")

## Step 5 — Plots

In [ ]:
ycol = "RAD21_consensus_pct" if rad_cols else None
if ycol:
    d = res[[X_COLUMN, ycol, "class", "num", rad_cols[0]]].dropna(subset=[X_COLUMN, ycol])
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
    for ax, (yc, ylab, logy) in zip(axes, [
            (rad_cols[0], f"RAD21 fold-enrichment\n({rad_cols[0].split('.')[0]})", True),
            (ycol, "RAD21 consensus rank percentile", False)]):
        for cls, g in d.groupby("class"):
            ax.scatter(g[X_COLUMN], g[yc], s=70, alpha=0.9, label=cls,
                       color=CLASS_COLOR.get(cls, "#52514e"),
                       edgecolor="white", linewidth=1.5, zorder=3)
        for _, r in d.iterrows():
            ax.annotate(int(r.num), (r[X_COLUMN], r[yc]), fontsize=7,
                        color="#52514e", xytext=(6, 4), textcoords="offset points")
        v = d[[X_COLUMN, yc]].dropna()
        if len(v) >= 4:
            rho, p = stats.spearmanr(v[X_COLUMN], v[yc])
            z = np.polyfit(v[X_COLUMN], np.log10(v[yc]) if logy else v[yc], 1)
            xs = np.linspace(v[X_COLUMN].min(), v[X_COLUMN].max(), 50)
            ys = np.polyval(z, xs)
            ax.plot(xs, 10**ys if logy else ys, color="#52514e",
                    lw=1.5, ls="--", alpha=0.6, zorder=2, label="OLS fit")
            ax.set_title(f"rho = {rho:+.2f},  p = {p:.3f},  n = {len(v)}",
                         fontsize=9, color="#0b0b0b")
        if logy:
            ax.set_yscale("log")
        ax.set_xlabel(f"Optical tweezers  ({X_COLUMN})")
        ax.set_ylabel(ylab)
    axes[0].legend(fontsize=8)
    fig.suptitle("Cohesin occupancy vs. tweezers measurement",
                 fontsize=11, y=1.04, color="#0b0b0b")
    fig.text(0.5, -0.04, "rho is Spearman; the dashed line is an ordinary least-squares "
             "fit. At small n the two can disagree in sign — read rho.",
             ha="center", fontsize=7.5, color="#52514e")
    fig.tight_layout()
    plt.show()
else:
    print("Configure a RAD21 bigWig first.")

In [ ]:
# RAD21 vs CTCF — shows how much of the RAD21 axis is just CTCF
if rad_cols and ctcf_cols:
    d = res[[rad_cols[0], ctcf_cols[0], "class", "num"]].dropna()
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    for cls, g in d.groupby("class"):
        ax.scatter(g[ctcf_cols[0]], g[rad_cols[0]], s=70, alpha=0.9, label=cls,
                   color=CLASS_COLOR.get(cls, "#52514e"),
                   edgecolor="white", linewidth=1.5, zorder=3)
    for _, r in d.iterrows():
        ax.annotate(int(r.num), (r[ctcf_cols[0]], r[rad_cols[0]]), fontsize=7,
                    color="#52514e", xytext=(6, 4), textcoords="offset points")
    rho, p = stats.spearmanr(d[ctcf_cols[0]], d[rad_cols[0]])
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("CTCF fold-enrichment"); ax.set_ylabel("RAD21 fold-enrichment")
    ax.set_title(f"RAD21 vs CTCF:  rho = {rho:+.2f}", fontsize=9)
    ax.legend(title="class", fontsize=8, title_fontsize=8)
    fig.tight_layout(); plt.show()

## Step 6 — Export

In [ ]:
res.to_csv("rad21_signal_at_motifs.csv", index=False)
if not ct.empty:
    ct.to_csv("correlation_results.csv", index=False)
with open("motifs_hg38.bed", "w") as f:
    for _, r in motifs.iterrows():
        f.write(f"{r.chrom}\t{r.start}\t{r.end}\t{r['name']}\t0\t{r.strand}\n")
print("wrote rad21_signal_at_motifs.csv, correlation_results.csv, motifs_hg38.bed")